In [13]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

In [14]:
class SimplePhishingDetector:
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        
    def train(self, csv_file):
        """Train the model - much simpler!"""
        print("Loading data...")
        df = pd.read_csv(csv_file)
        
        # Features and target
        feature_cols = [
            "length_url", "length_hostname", "nb_dots", "nb_hyphens", "nb_at",
            "nb_qm", "nb_and", "nb_eq", "nb_slash", "nb_colon", "nb_subdomains",
            "prefix_suffix", "tld_in_path", "tld_in_subdomain", "shortening_service",
            "path_extension", "char_repeat", "longest_word_host", "longest_word_path",
            "ratio_digits_url", "ratio_digits_host", "whois_registered_domain",
            "domain_registration_length", "domain_age", "dns_record"
        ]
        
        X = df[feature_cols].fillna(0)  # Simple: just fill missing with 0
        y = df['status']
        
        # Encode labels
        y_encoded = self.label_encoder.fit_transform(y)
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
        
        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        # Train XGBoost
        print("Training model...")
        self.model = xgb.XGBClassifier(
            max_depth=5,
            learning_rate=0.1,
            n_estimators=200,
            random_state=42
        )
        
        self.model.fit(X_train_scaled, y_train)
        
        # Evaluate
        y_pred = self.model.predict(X_test_scaled)
        accuracy = accuracy_score(y_test, y_pred)
        
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Classes: {self.label_encoder.classes_}")
        
        return accuracy
    
    def save_model(self, filename="phishing_model"):
        """Save model the normal way"""
        if self.model is None:
            print("No model to save!")
            return False
            
        try:
            # Save XGBoost model
            self.model.save_model(f"{filename}.json")
            
            # Save scaler and encoder with pickle (simple!)
            with open(f"{filename}_scaler.pkl", 'wb') as f:
                pickle.dump(self.scaler, f)
                
            with open(f"{filename}_encoder.pkl", 'wb') as f:
                pickle.dump(self.label_encoder, f)
            
            print(f"Model saved as {filename}.json")
            print(f"Scaler saved as {filename}_scaler.pkl")  
            print(f"Encoder saved as {filename}_encoder.pkl")
            return True
            
        except Exception as e:
            print(f"Error saving: {e}")
            return False
    
    def load_model(self, filename="phishing_model"):
        """Load model the normal way"""
        try:
            # Load XGBoost model
            self.model = xgb.XGBClassifier()
            self.model.load_model(f"{filename}.json")
            
            # Load scaler and encoder
            with open(f"{filename}_scaler.pkl", 'rb') as f:
                self.scaler = pickle.load(f)
                
            with open(f"{filename}_encoder.pkl", 'rb') as f:
                self.label_encoder = pickle.load(f)
            
            print(f"Model loaded from {filename}.json")
            return True
            
        except Exception as e:
            print(f"Error loading: {e}")
            return False
    
    def predict(self, url_features):
        """Make prediction"""
        if self.model is None:
            print("No model loaded!")
            return None
        
        try:
            # Convert to DataFrame if it's a dict
            if isinstance(url_features, dict):
                # Ensure all 25 features are present
                required_features = [
                    "length_url", "length_hostname", "nb_dots", "nb_hyphens", "nb_at",
                    "nb_qm", "nb_and", "nb_eq", "nb_slash", "nb_colon", "nb_subdomains",
                    "prefix_suffix", "tld_in_path", "tld_in_subdomain", "shortening_service",
                    "path_extension", "char_repeat", "longest_word_host", "longest_word_path",
                    "ratio_digits_url", "ratio_digits_host", "whois_registered_domain",
                    "domain_registration_length", "domain_age", "dns_record"
                ]
                
                # Fill missing features with 0
                for feature in required_features:
                    if feature not in url_features:
                        url_features[feature] = 0
                
                # Convert to list in correct order
                features_list = [url_features[f] for f in required_features]
                features_array = np.array([features_list])
            else:
                features_array = np.array([url_features])
            
            # Scale features
            features_scaled = self.scaler.transform(features_array)
            
            # Predict
            prediction = self.model.predict(features_scaled)[0]
            probabilities = self.model.predict_proba(features_scaled)[0]
            
            # Convert back to original labels
            predicted_label = self.label_encoder.inverse_transform([prediction])[0]
            
            return {
                "prediction": predicted_label,
                "is_phishing": predicted_label == "phishing",
                "confidence": float(max(probabilities)),
                "probabilities": {
                    self.label_encoder.classes_[0]: float(probabilities[0]),
                    self.label_encoder.classes_[1]: float(probabilities[1])
                }
            }
            
        except Exception as e:
            print(f"Prediction error: {e}")
            return None

In [16]:
# Simple usage
if __name__ == "__main__":
    # Train and save
    detector = SimplePhishingDetector()
    detector.train("dataset_phishing.csv")
    detector.save_model("my_simple_model")
    
    # Test prediction
    test_features = {
        'length_url': 87, 'length_hostname': 20, 'nb_dots': 3, 'nb_hyphens': 1,
        'nb_at': 0, 'nb_qm': 1, 'nb_and': 2, 'nb_eq': 1, 'nb_slash': 4,
        'nb_colon': 1, 'nb_subdomains': 2, 'prefix_suffix': 0, 'tld_in_path': 0,
        'tld_in_subdomain': 0, 'shortening_service': 0, 'path_extension': 1,
        'char_repeat': 2, 'longest_word_host': 8, 'longest_word_path': 6,
        'ratio_digits_url': 0.15, 'ratio_digits_host': 0.1,
        'whois_registered_domain': 1, 'domain_registration_length': 365,
        'domain_age': 1000, 'dns_record': 1
    }
    
    result = detector.predict(test_features)
    print("\nPrediction:", result)
    
    print("\n" + "="*50)
    print("TESTING MODEL LOADING")
    print("="*50)
    
    # Test loading saved model
    new_detector = SimplePhishingDetector()
    new_detector.load_model("my_simple_model")
    
    result2 = new_detector.predict(test_features)
    print("SINGLE URL prediction from loaded model:", result2)

Loading data...
Training model...
Accuracy: 0.9208
Classes: ['legitimate' 'phishing']
Model saved as my_simple_model.json
Scaler saved as my_simple_model_scaler.pkl
Encoder saved as my_simple_model_encoder.pkl

Prediction: {'prediction': 'phishing', 'is_phishing': True, 'confidence': 0.9975726008415222, 'probabilities': {'legitimate': 0.002427399158477783, 'phishing': 0.9975726008415222}}

TESTING MODEL LOADING
Model loaded from my_simple_model.json
SINGLE URL prediction from loaded model: {'prediction': 'phishing', 'is_phishing': True, 'confidence': 0.9975726008415222, 'probabilities': {'legitimate': 0.002427399158477783, 'phishing': 0.9975726008415222}}
